<a href="https://colab.research.google.com/github/mmuputisi/Adv-Py_Data_Analysis/blob/main/Supply%20Chain%20Optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# Supply Chain Optimizer - Full Colab Version
# =========================

# --- 1️⃣ Install required packages ---
!pip install --quiet openrouteservice pandas numpy matplotlib seaborn geopandas scikit-learn tqdm contextily plotly ortools

# --- 2️⃣ Imports ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import KMeans
import contextily as ctx
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timezone
import pickle
import random

# ORS imports
try:
    import openrouteservice
    from openrouteservice.exceptions import ApiError
except ImportError:
    print("OpenRouteService not available; will fallback to Haversine distances.")
    openrouteservice = None

sns.set(style="whitegrid")
print("Imports ready")

# --- 3️⃣ Mount Google Drive safely ---
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

# --- 4️⃣ File paths ---
EXCEL_PATH = "/content/drive/MyDrive/Colab Notebooks/health facilities list.xlsx"
CACHE_PATH = "/content/drive/MyDrive/supply_chain_cache/ors_cache.pkl"
OUTPUT_FOLDER = "/content/drive/MyDrive/supply_chain_outputs/"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(f"Excel file not found at {EXCEL_PATH}")

# --- 5️⃣ Load facilities ---
df_facilities = pd.read_excel(EXCEL_PATH, header=1)
df_facilities.columns = df_facilities.iloc[0]
df_facilities = df_facilities.drop(0).reset_index(drop=True)
df_facilities = df_facilities.iloc[:, 3:]  # Drop first 3 empty columns
df_facilities.columns = [str(col).strip() for col in df_facilities.columns]

df_facilities['longitude'] = pd.to_numeric(df_facilities['Longitude'], errors='coerce')
df_facilities['latitude'] = pd.to_numeric(df_facilities['Latitude'], errors='coerce')
df_facilities['frequency'] = pd.to_numeric(df_facilities.get('Frequency', 1), errors='coerce')
df_facilities = df_facilities.dropna(subset=['longitude','latitude'])

df_facilities.rename(columns={
    'State':'state',
    'Facility name':'facility_name',
    'Facility type':'facility_type'
}, inplace=True)

print(f"Loaded {len(df_facilities)} facilities across {df_facilities['state'].nunique()} states")

# --- 6️⃣ Load city/capital data ---
cities_data = {
    'State': ['Abia', 'Adamawa', 'Akwa-Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 'Borno',
              'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Enugu', 'Gombe', 'Imo', 'Jigawa',
              'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger',
              'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe',
              'Zamfara', 'FCT'],
    'Capital': ['Umuahia', 'Yola', 'Uyo', 'Awka', 'Bauchi', 'Yenagoa', 'Makurdi', 'Maiduguri',
                'Calabar', 'Asaba', 'Abakaliki', 'Benin City', 'Ado-Ekiti', 'Enugu', 'Gombe',
                'Owerri', 'Dutse', 'Kaduna', 'Kano', 'Katsina', 'Birnin Kebbi', 'Lokoja', 'Ilorin',
                'Ikeja', 'Lafia', 'Minna', 'Abeokuta', 'Akure', 'Osogbo', 'Ibadan', 'Jos',
                'Port Harcourt', 'Sokoto', 'Jalingo', 'Damaturu', 'Gusau', 'Abuja'],
    'Lat': [5.52491, 9.20839, 5.05127, 6.21269, 10.31032, 4.92675, 7.73375, 11.84692, 4.95893,
            6.19824, 6.32485, 6.33815, 7.62329, 6.44132, 10.28969, 5.48363, 11.75618, 10.52641,
            12.00012, 12.99082, 12.45389, 7.79688, 8.49664, 6.59651, 8.4939, 9.61524, 7.15571,
            7.25256, 7.77104, 7.37756, 9.92849, 4.77742, 13.06269, 8.89367, 11.74697, 12.17024,
            9.05785],
    'Long': [7.49461, 12.48146, 7.9335, 7.07199, 9.84388, 6.26764, 8.52139, 13.15712, 8.32695,
             6.73187, 8.11368, 5.62575, 5.22087, 7.49883, 11.16729, 7.03325, 9.33896, 7.43879,
             8.51672, 7.60177, 4.1975, 6.74048, 4.54214, 3.34205, 8.51532, 6.54776, 3.34509,
             5.19312, 4.55698, 3.90591, 8.89212, 7.0134, 5.24322, 11.3596, 11.96083, 6.66412,
             7.49508]
}
df_cities = pd.DataFrame(cities_data)
df_cities.rename(columns={'Lat':'latitude','Long':'longitude','Capital':'city','State':'state'}, inplace=True)

# --- 7️⃣ Haversine distance fallback ---
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1,lon1,lat2,lon2])
    dlat, dlon = lat2-lat1, lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R*2*np.arcsin(np.sqrt(a))

# --- 8️⃣ ORS Distance with caching ---
class ORSDistanceCache:
    def __init__(self, api_key=None, cache_path=CACHE_PATH):
        self.client = openrouteservice.Client(key=api_key) if api_key else None
        self.cache_path = cache_path
        self.cache = self._load_cache()

    def _load_cache(self):
        if os.path.exists(self.cache_path):
            with open(self.cache_path,'rb') as f:
                return pickle.load(f)
        return {}

    def save_cache(self):
        with open(self.cache_path,'wb') as f:
            pickle.dump(self.cache,f)

    def get_distance(self, from_coord, to_coord):
        k = (from_coord, to_coord)
        if k in self.cache:
            return self.cache[k]
        # fallback to haversine if ORS unavailable
        if self.client:
            try:
                res = self.client.directions([from_coord,to_coord], profile='driving-car', format='json')
                dist = res['routes'][0]['summary']['distance']/1000  # km
            except ApiError:
                dist = haversine_distance(from_coord[1],from_coord[0],to_coord[1],to_coord[0])
        else:
            dist = haversine_distance(from_coord[1],from_coord[0],to_coord[1],to_coord[0])
        self.cache[k] = dist
        return dist

# --- 9️⃣ Assign facilities to warehouses ---
def assign_facilities(facilities, warehouses, ors_cache=None):
    assignments = []
    for _, f in tqdm(facilities.iterrows(), total=len(facilities)):
        min_dist = float('inf')
        nearest_wh = None
        for _, w in warehouses.iterrows():
            dist = ors_cache.get_distance((f['longitude'], f['latitude']), (w['longitude'], w['latitude'])) if ors_cache else haversine_distance(f['latitude'], f['longitude'], w['latitude'], w['longitude'])
            if dist < min_dist:
                min_dist, nearest_wh = dist, w['city']
        assignments.append({'warehouse_city':nearest_wh,'distance_km':min_dist,'transport_cost':min_dist*0.5})
    return pd.concat([facilities.reset_index(drop=True), pd.DataFrame(assignments)], axis=1)

# --- 1️⃣0️⃣ Optimization scenarios ---
scenario_warehouses = [1,3,5,7,10]  # number of regional warehouses
EXCLUDE_HUB = False
ORS_API_KEY = "YOUR_ORS_API_KEY_HERE"  # Replace with your ORS key
ors_cache = ORSDistanceCache(api_key=ORS_API_KEY)

scenarios = {}
for n_wh in scenario_warehouses:
    print(f"Running scenario: {n_wh} regional warehouses")
    # KMeans to choose cluster centroids
    kmeans = KMeans(n_clusters=n_wh, random_state=42, n_init=10)
    kmeans.fit(df_facilities[['latitude','longitude']])
    centroids = kmeans.cluster_centers_
    # select nearest city to centroid
    selected_wh = []
    for lat, lon in centroids:
        nearest_city = df_cities.iloc[((df_cities['latitude']-lat)**2+(df_cities['longitude']-lon)**2).argmin()]
        if nearest_city['city'] not in selected_wh:
            selected_wh.append(nearest_city['city'])
    wh_df = df_cities[df_cities['city'].isin(selected_wh)]
    # assign facilities
    result_df = assign_facilities(df_facilities, wh_df, ors_cache)
    total_dist = result_df['distance_km'].sum()
    avg_dist = result_df['distance_km'].mean()
    total_cost = result_df['transport_cost'].sum()
    max_dist = result_df['distance_km'].max()
    scenarios[n_wh] = {'warehouses':selected_wh,'total_dist':total_dist,'avg_dist':avg_dist,'total_cost':total_cost,'max_dist':max_dist,'assignments':result_df}
    print(f"Selected warehouses: {selected_wh}")
    print(f"Total dist={total_dist:.2f} km, avg={avg_dist:.2f}, cost=${total_cost:.2f}, max={max_dist:.2f} km")

ors_cache.save_cache()

# --- 1️⃣1️⃣ Scenario comparison plot ---
summary_df = pd.DataFrame({
    'Warehouses': list(scenarios.keys()),
    'Total Distance': [scenarios[n]['total_dist'] for n in scenarios],
    'Avg Distance': [scenarios[n]['avg_dist'] for n in scenarios],
    'Total Cost': [scenarios[n]['total_cost'] for n in scenarios],
    'Max Distance': [scenarios[n]['max_dist'] for n in scenarios]
})

plt.figure(figsize=(10,6))
for col in ['Total Distance','Avg Distance','Total Cost','Max Distance']:
    plt.plot(summary_df['Warehouses'], summary_df[col], marker='o', label=col)
plt.xlabel("Number of Regional Warehouses")
plt.ylabel("Value")
plt.title("Scenario Comparison")
plt.legend()
plt.grid(True)
plt.show()

print("All scenarios computed. ORS cache saved.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.7/27.7 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 42.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.31.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.31.1 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.31.1 which is incompatible.
Imports ready
Mounted at /content/drive
Loaded 19693 facil

100%|██████████| 19693/19693 [1:26:32<00:00,  3.79it/s]


Selected warehouses: ['Abuja']
Total dist=6973105.56 km, avg=354.09, cost=$3486552.78, max=850.56 km
Running scenario: 3 regional warehouses


 12%|█▏        | 2349/19693 [30:32<3:44:50,  1.29it/s]